# Economics Lab: Logistic Regression & the Decision Boundary
## Application — Mortgage Default Risk (SKELETON — fill in the `# TODO`s)

*Adapted from a machine-learning "decision boundary" exercise, re-cast as an economics / credit-risk problem. Work through the TODOs in order — each one builds on the last.*


## Goals
In this lab you will:
- Build a small two-feature credit-risk dataset (Debt-to-Income ratio, Loan-to-Value ratio) and a binary outcome (mortgage **default** vs. **repaid**).
- Refresh how the sigmoid function turns a linear score into a probability.
- Derive and plot the **decision boundary** of a logistic regression classifier by hand, given its parameters.
- Interpret the boundary and the coefficients in economic terms (which combinations of leverage are "risky"?).
- Extend the idea to a **non-linear** boundary using polynomial terms, and discuss when a linear boundary is/isn't appropriate for economic data.


## Setup
Run this cell — it defines small plotting/math helpers you'll reuse throughout (nothing to fill in here).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    """Numerically stable sigmoid / logistic function."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def plot_data(X, y, ax, pos_label="Default (y=1)", neg_label="Repaid (y=0)", s=90):
    """Scatter-plot a 2-feature binary-outcome dataset."""
    y = np.array(y).reshape(-1)
    pos = y == 1
    neg = y == 0
    ax.scatter(X[pos, 0], X[pos, 1], marker='x', s=s, c='crimson', linewidths=2, label=pos_label)
    ax.scatter(X[neg, 0], X[neg, 1], marker='o', s=s, facecolors='none',
               edgecolors='steelblue', linewidths=2, label=neg_label)
    ax.legend(loc='best')

def draw_vthresh(ax, x):
    """Shade g(z)<0.5 vs g(z)>=0.5 either side of z=x on a sigmoid plot."""
    ylim, xlim = ax.get_ylim(), ax.get_xlim()
    ax.fill_between([xlim[0], x], ylim[1], alpha=0.15, color='steelblue')
    ax.fill_between([x, xlim[1]], ylim[1], alpha=0.15, color='crimson')
    ax.axvline(x, color='k', ls='--', lw=1)
    ax.set_xlim(xlim); ax.set_ylim(ylim)

def plot_linear_boundary(w, b, ax, x_range=(0, 4), color='seagreen', label='Decision boundary'):
    """Plot w0*x0 + w1*x1 + b = 0 and shade the predicted-0 region."""
    x0 = np.linspace(x_range[0], x_range[1], 200)
    x1 = -(w[0] * x0 + b) / w[1]
    ax.plot(x0, x1, c=color, lw=2, label=label)
    ax.fill_between(x0, x1, ax.get_ylim()[0], alpha=0.12, color=color)
    return x0, x1


## 1. The dataset

Suppose a bank has six recent mortgage applicants. For each one we record two standardized risk indices:

| Applicant | DTI index $x_0$ | LTV index $x_1$ | Outcome $y$ |
|---|---|---|---|
| 1 | 0.5 | 1.5 | 0 (repaid) |
| 2 | 1.0 | 1.0 | 0 (repaid) |
| 3 | 1.5 | 0.5 | 0 (repaid) |
| 4 | 3.0 | 0.5 | 1 (default) |
| 5 | 2.0 | 2.0 | 1 (default) |
| 6 | 1.0 | 2.5 | 1 (default) |

- $x_0$ = **Debt-to-Income (DTI) index** — higher means the household is servicing more debt relative to income.
- $x_1$ = **Loan-to-Value (LTV) index** — higher means the loan is large relative to the collateral value.
- $y=1$ means the loan **defaulted**; $y=0$ means it was **repaid**.

**TODO 1:** Build `X` (shape `(6,2)`) and `y` (shape `(6,1)`) from the table above.

In [ ]:
# TODO 1: create the numpy arrays X and y from the table above
X = None  # TODO — shape (6, 2): columns are [DTI index, LTV index]
y = None  # TODO — shape (6, 1): 0 = repaid, 1 = default

print("X shape:", None if X is None else X.shape)
print("y shape:", None if y is None else y.shape)


### 1.1 Plot the data

**TODO 2:** Use the `plot_data` helper to scatter-plot `X`/`y` on the axes provided. Label the axes `DTI index` and `LTV index`.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# TODO 2: call plot_data(X, y, ax) and set axis labels / limits
# plot_data(...)
# ax.set_xlabel(...)
# ax.set_ylabel(...)

ax.axis([0, 4, 0, 3.5])
plt.show()


## 2. The logistic regression model

The bank's risk team has already fit a logistic regression model (you'll learn *how* to fit one later in the course). It has the form

$$f(\mathbf{x}) = g(w_0 x_0 + w_1 x_1 + b), \qquad g(z) = \frac{1}{1+e^{-z}}$$

and the fitted parameters are $b=-3,\ w_0=1,\ w_1=1$, so

$$f(\mathbf{x}) = g(x_0 + x_1 - 3)$$

$f(\mathbf{x})$ is interpreted as the **predicted probability of default** given the applicant's DTI and LTV indices.

### Refresher: from probability to a 0/1 decision

We classify:
- predict $y=1$ (default) if $f(\mathbf{x}) \ge 0.5$
- predict $y=0$ (repaid) if $f(\mathbf{x}) < 0.5$

Since $g(z)\ge 0.5 \iff z \ge 0$, this is the same as:
- predict $y=1$ if $w_0x_0+w_1x_1+b \ge 0$
- predict $y=0$ if $w_0x_0+w_1x_1+b < 0$

**TODO 3:** Plot the sigmoid function for $z\in[-10,10]$ and use `draw_vthresh` to shade the $z\ge0$ region.

In [ ]:
z = np.arange(-10, 11)

fig, ax = plt.subplots(1, 1, figsize=(5, 3))
# TODO 3: plot z vs sigmoid(z), then call draw_vthresh(ax, 0)
# ax.plot(...)
# draw_vthresh(...)

ax.set_title("Sigmoid function")
ax.set_xlabel("z")
ax.set_ylabel("g(z)")
plt.show()


## 3. Deriving and plotting the decision boundary

The **decision boundary** is the set of points where the model is exactly indifferent: $w_0x_0+w_1x_1+b = 0$.

For our model that's $x_0 + x_1 - 3 = 0$, i.e. $x_1 = 3 - x_0$.

**TODO 4:**
1. Compute `x1_boundary = 3 - x0` for `x0 = np.arange(0, 6)`.
2. Plot the boundary line.
3. Shade the region **below** the line (predicted repaid, $y=0$).
4. Overlay the original data points with `plot_data`.

In [ ]:
x0 = np.arange(0, 6)
x1_boundary = None  # TODO 4a: x1 = 3 - x0

fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# TODO 4b: plot the boundary line (x0, x1_boundary)
# TODO 4c: shade below the line with ax.fill_between
# TODO 4d: overlay the data with plot_data(X, y, ax)

ax.axis([0, 4, 0, 3.5])
ax.set_xlabel("DTI index")
ax.set_ylabel("LTV index")
plt.show()


**TODO 5 (short answer — write in this cell):**

1. In plain economic language, what does the decision boundary line represent?
2. Applicant 5 has DTI = 2.0, LTV = 2.0. Which side of the boundary are they on, and does that match the observed outcome in the table?
3. Two applicants can have very different DTI/LTV combinations and still land on the same side of the boundary (e.g. DTI=3, LTV=0.2 vs DTI=0.2, LTV=3). What does that imply about how this model treats "leverage from debt" vs. "leverage from loan size"? Is that a realistic assumption?

*(Your answers here)*


## 4. Extension: a non-linear boundary (optional / stretch)

Credit-risk relationships are often **not** well described by a straight line — e.g. default risk may accelerate sharply once DTI passes some threshold (a convex relationship), rather than rising at a constant rate.

Suppose instead the risk team fits: $f(\mathbf{x}) = g(x_0^2 + x_1 - 1)$.

**TODO 6:**
1. Solve for the boundary: $x_1 = 1 - x_0^2$.
2. Plot it over $x_0 \in [-2, 2]$ together with the original data (don't worry that the data was generated for the linear model — just practice the mechanics).
3. In a markdown cell, describe in words how this boundary's *shape* differs from the straight-line case, and what kind of economic story (e.g. "risk accelerates non-linearly with leverage") would justify using it.

In [ ]:
x0_nl = np.linspace(-2, 2, 200)
x1_nl_boundary = None  # TODO 6a: 1 - x0_nl**2

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
# TODO 6b: plot the curve, shade a region, and overlay plot_data(X, y, ax)

ax.set_xlabel("DTI index")
ax.set_ylabel("LTV index")
plt.show()


*(Your written comparison for TODO 6.3 here)*

## Wrap-up

You practiced:
- Encoding a small economic dataset for classification
- Refreshing the sigmoid / 0.5-threshold logic
- Deriving a **linear** decision boundary from logistic-regression coefficients by hand
- Reasoning about what the boundary and coefficients mean economically
- Sketching a **non-linear** boundary and thinking about when it's justified

➡️ Next: open the **Cheatsheet** notebook for the formulas/interpretation reference, or the **Template** notebook to apply this to a *different* economics classification problem of your choice.
